In [1]:
import asyncio
import json
import socket
import httpx
from langgraph_sdk import get_client
from langchain_core.output_parsers import StrOutputParser
import os
from dotenv import load_dotenv, find_dotenv

# Carga el .env del proyecto (busca desde el cwd) y pisa una GOOGLE_API_KEY vieja del kernel.
load_dotenv(find_dotenv(usecwd=True), override=True)
if not os.environ.get("GOOGLE_API_KEY"):
    raise RuntimeError("Falta GOOGLE_API_KEY: pon una API key valida en el .env del proyecto")

LANGGRAPH_URL = "http://127.0.0.1:2025"  # agente 1; el agente 2 corre en el puerto 2026
A2A_HOST = "127.0.0.1"
A2A_PORT = 2025  # agente 1; para el segundo agente usa 2026
A2A_URL = f"http://{A2A_HOST}:{A2A_PORT}"
A2A_RPC_PATH = "/a2a/"

# No enviamos automáticamente claves de LangSmith al servidor local.
lg = get_client(url=LANGGRAPH_URL, api_key=None, timeout=120)

def mostrar(valor):
    print(json.dumps(valor, indent=2, ensure_ascii=False, default=str))

# Encontrar el Id de tu agente

In [3]:
# El formato se fija para que el listado no dependa del valor por defecto del SDK.
asistentes = await lg.assistants.search(limit=100, response_format="array")
for asistente in asistentes:
    print({
        "name": asistente.get("name"),
        "graph_id": asistente.get("graph_id"),
        "assistant_id": asistente.get("assistant_id"),
    })
if not asistentes:
    raise RuntimeError("El servidor no devolvió asistentes. Revisa langgraph.json.")

{'name': 'agente_rodrigo', 'graph_id': 'agente_rodrigo', 'assistant_id': '097095cf-e5e1-52ab-b240-14eaa9b5d64c'}
{'name': 'agente_viajes', 'graph_id': 'agente_viajes', 'assistant_id': 'f4c036e2-e856-5746-a4ec-f64eb476af3d'}


In [4]:
ASSISTANT_ID = '097095cf-e5e1-52ab-b240-14eaa9b5d64c'

# Comunicacion directo con el agente

In [5]:
OUTPUT_KEY = None  # Ejemplo: "respuesta" si tu salida no utiliza messages.

def crear_entrada(texto):
    return {"messages": [{"role": "human", "content": texto}]}


# Extraer respuesta del json gigantesco del grafo
def extraer_texto(estado):
    if not isinstance(estado, dict):
        raise ValueError("La salida no es un diccionario; adapta extraer_texto a tu grafo.")
    if estado.get("__interrupt__"):
        raise RuntimeError("Tu grafo quedó pausado: resuelve el interrupt antes de responder.")
    if OUTPUT_KEY:
        contenido = estado[OUTPUT_KEY]
    else:
        # No tomamos el mensaje humano como si fuera una respuesta del agente.
        mensajes = estado.get("messages", [])
        respuesta = next(
            (m for m in reversed(mensajes)
             if m.get("type", m.get("role")) in ("ai", "assistant")),
            None,
        )
        if respuesta is None:
            raise ValueError("No hay respuesta ai/assistant. Configura OUTPUT_KEY.")
        contenido = respuesta.get("content", "")
    if isinstance(contenido, list):
        contenido = "\n".join(
            bloque if isinstance(bloque, str) else bloque.get("text", "")
            for bloque in contenido
            if isinstance(bloque, (str, dict))
        )
    if not isinstance(contenido, str) or not contenido.strip():
        raise ValueError("La salida no tiene texto. Adapta extraer_texto a tu agente.")
    return contenido


async def consultar_langgraph(texto, thread_id=None):
    estado = await lg.runs.wait(
        thread_id,
        ASSISTANT_ID,
        input=crear_entrada(texto),
    )
    return extraer_texto(estado)

### Pregunta 1 · Personalidad

In [6]:
PREGUNTA = "Preséntate en pocas líneas: ¿quién eres, cuál es tu especialidad y en qué me puedes ayudar?"
estado = await lg.runs.wait(None, ASSISTANT_ID, input=crear_entrada(PREGUNTA))
print(extraer_texto(estado))

¡Hola! Qué gusto saludarte. 

Soy Rodrigo, especialista en arquitectura backend con Java y Spring Boot (+15 años en el ruedo) y apasionado por la construcción de agentes de IA. Eso sí, déjame serte totalmente honesto: aunque estudio con todo para dominar los agentes inteligentes, mi experiencia fuerte y en producción es el desarrollo backend tradicional; ¡con los agentes voy aprendiendo y experimentando paso a paso!

¿En qué te puedo ayudar? 
1. A auditar, diseñar y mejorar microservicios y APIs REST (contratos, resiliencia, observabilidad).
2. A entender y diseñar sistemas basados en agentes de IA (tool calling, RAG, orquestación).
3. ¡Y si por ahí te interesa la música norteña o vallenata, también te puedo orientar vendiendo algún acordeón Hohner por YouTube! 

¿Qué proyecto traes entre manos hoy?


### Pregunta 2 · Conocimientos y skills

In [7]:
# Pregunta 2 · Conocimientos y skills (usa el caso de sus apuntes y una habilidad)
PREGUNTA = ("Estoy postulando a un puesto backend Java en el equipo del carrito de compras "
            "de Falabella. Usa el caso de tus apuntes y proponme una mejora concreta para ese equipo.")
estado = await lg.runs.wait(None, ASSISTANT_ID, input=crear_entrada(PREGUNTA))
print(extraer_texto(estado))

¡Hola! Qué excelente noticia que estés postulando al equipo del carrito de compras de Falabella. ¡Es un desafío gigante y súper entretenido desde el punto de vista de la ingeniería de backend!

Primero, saquemos a la mesa lo que dicen mis apuntes textuales sobre este caso:
> *"Caso real que me marcó: el carro de compras de Falabella. Contexto: alto tráfico multi-país (varios países, miles de usuarios concurrentes). APIs REST entre servicios internos. Caché con Redis para aliviar la base de datos y bajar latencia. Arquitectura multi-tenant en GCP con Docker y Kubernetes. Dashboards de observabilidad para monitorear en tiempo real. Qué aprendí: separar bien los contratos, dimensionar el caché, diseñar para fallar de forma controlada y medir todo lo que importa."*

---

### Mi propuesta de mejora concreta (Basada en mi experiencia de +15 años en Java/Spring Boot)

Tomando en cuenta ese escenario de altísimo tráfico multi-país y el uso de Redis para el carrito, te sugiero proponer una mejo

# Memoria

In [7]:
# Pregunta 3 · Memoria (dos turnos, mismo thread_id). Usa el ASSISTANT_ID de la celda 4.
thread = await lg.threads.create()
thread_id = thread["thread_id"]
print(await consultar_langgraph("Mi objetivo es construir agentes de IA confiables sobre mi base backend.", thread_id=thread_id))
print(await consultar_langgraph("¿Qué objetivo te acabo de contar?", thread_id=thread_id))

¡Listo, ya le he echado un vistazo a mis apuntes! 

Según mis apuntes, en la sección de **Agentes de IA**, los pilares fundamentales que debemos dominar son el **tool calling** (para que los agentes interactúen con tus sistemas), la **orquestación multiagente**, **RAG** (para darles contexto de tus datos) y el **Model Context Protocol (MCP)**. 

*(Ojo: esto es lo que tengo registrado textualmente en mis apuntes).*

Ahora bien, conectándolo con mi experiencia real: **mi fuerte absoluto es el backend** (Java, Spring Boot, APIs REST, resiliencia, observabilidad). Aunque estoy certificado y estudio a fondo la teoría de agentes de IA, quiero serte completamente honesto: **no tengo experiencia en producción construyendo sistemas complejos de agentes de IA**. Mis recomendaciones irán desde la perspectiva de un arquitecto backend tradicional que ve a los agentes como un cliente especial y muy demandante para tus APIs.

Para empezar a diseñar esta arquitectura juntos, cuéntame un poco más: 
¿Qu

Me acabas de contar que tu objetivo es **construir agentes de IA confiables sobre tu base backend**. 

*(Esto lo tengo fresquito porque me lo escribiste en tu primer mensaje).* 

¿Hay algún detalle adicional de ese objetivo que quieras que profundicemos, o pasamos a ver cómo estructurar las APIs o el tool calling para lograrlo?


# Como se ve esto en langraph

In [8]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class EstadoCoordinador(TypedDict):
    pregunta: str
    respuesta: str

consulta_remota = consultar_langgraph

async def delegar(estado: EstadoCoordinador):
    respuesta = await consulta_remota(estado["pregunta"])
    return {"respuesta": respuesta}

builder = StateGraph(EstadoCoordinador)
builder.add_node("delegar", delegar)

builder.add_edge(START, "delegar")
builder.add_edge("delegar", END)

coordinador = builder.compile()

resultado = await coordinador.ainvoke({"pregunta": "Hola que puedes hacer?"})
print(resultado["respuesta"])

¡Hola! Qué gusto saludarte. 

Soy Rodrigo. Llevo más de 15 años metido de cabeza en el desarrollo backend con **Java y Spring Boot**, diseñando microservicios y APIs REST robustas. Además, como tengo un Magíster en Data Science, me apasiona todo el mundo de la **construcción de agentes de IA** (tool calling, orquestación, RAG, protocolos como MCP). 

Eso sí, seamos transparentes desde el minuto uno: aunque estudio mucho sobre agentes de IA y estoy haciendo una certificación, **no tengo experiencia profesional en producción con sistemas de agentes**. Mi fuerte absoluto es el backend tradicional. 

Ah, y como dato de color (¡y de negocio!), también me apasiona y sé un montón sobre **vender acordeones Hohner por YouTube**.

En resumen, te puedo ayudar con:
1. **Auditar y mejorar diseños de microservicios o APIs REST** (contratos, tolerancia a fallos, caché, observabilidad).
2. **Explicar y diseñar arquitecturas para agentes de IA** (basándome en lo que estudio y mis apuntes personales).
3

In [9]:
print(coordinador.get_graph().draw_mermaid())            # texto, siempre funciona


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	delegar(delegar)
	__end__([<p>__end__</p>]):::last
	__start__ --> delegar;
	delegar --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



# Vamos a Agent2Agent.

In [8]:
# pip install a2a-sdk
from a2a.client import A2ACardResolver, ClientConfig, create_client
from a2a.types import Role, SendMessageRequest
from a2a.helpers import get_message_text
import httpx 
from google.protobuf.json_format import MessageToDict

http_a2a = httpx.AsyncClient(timeout=120, trust_env=False)
resolver = A2ACardResolver(http_a2a, A2A_URL)
card_remota = await resolver.get_agent_card(
    http_kwargs={"params": {"assistant_id": ASSISTANT_ID}}
)
print("Ficha descargada:")
mostrar(MessageToDict(card_remota))

cliente_a2a = await create_client(
    card_remota,
    client_config=ClientConfig(
        httpx_client=http_a2a,
        streaming=False,
        supported_protocol_bindings=["JSONRPC"],
    ),
)
print("Cliente preparado con la Agent Card.")

await cliente_a2a.close()


Ficha descargada:
{
  "name": "agente_rodrigo",
  "description": "agente_rodrigo assistant",
  "supportedInterfaces": [
    {
      "url": "http://127.0.0.1:2025/a2a/097095cf-e5e1-52ab-b240-14eaa9b5d64c",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "1.0"
    }
  ],
  "version": "0.16.1",
  "capabilities": {
    "streaming": true,
    "pushNotifications": false,
    "extensions": [
      {
        "uri": "https://langchain.com/a2a/extensions/history-scope/v1",
        "description": "Choose task-only or full-context response history",
        "params": {
          "values": [
            "context",
            "task"
          ],
          "header": "LangGraph-A2A-History-Scope",
          "default": "context",
          "methods": [
            "SendMessage",
            "GetTask",
            "ListTasks"
          ]
        }
      }
    ]
  },
  "defaultInputModes": [
    "application/json",
    "text/plain"
  ],
  "defaultOutputModes": [
    "application/json",
    

## Envío por A2A (Message → Task → artifact)

In [9]:
# A2A: descargar la tarjeta y enviar un mensaje (Message -> Task -> artifact)
import httpx
from a2a.client import A2ACardResolver, ClientConfig, create_client
from a2a.types import Role, SendMessageRequest, TaskState
from a2a.helpers import new_text_message, get_message_text, get_artifact_text

http_a2a = httpx.AsyncClient(timeout=120, trust_env=False)
resolver = A2ACardResolver(http_a2a, A2A_URL)

card = await resolver.get_agent_card(
    http_kwargs={"params": {"assistant_id": ASSISTANT_ID}}
)
print("Tarjeta:", card.name, "->", card.supported_interfaces[0].url)

# Igual que en el supervisor (celda 17): fija la version de protocolo que entiende el cliente.
for interfaz in card.supported_interfaces:
    if interfaz.protocol_binding == "JSONRPC":
        interfaz.protocol_version = "0.3.0"

cliente = await create_client(
    card,
    client_config=ClientConfig(
        httpx_client=http_a2a,
        streaming=False,
        supported_protocol_bindings=["JSONRPC"],
    ),
)

request = SendMessageRequest(
    message=new_text_message(
        "¿Qué aprendiste en el carro de compras de Falabella?",
        role=Role.ROLE_USER,
    )
)

textos = []
async for evento in cliente.send_message(request):
    if evento.HasField("message"):
        print("Respuesta tipo mensaje")
        textos.append(get_message_text(evento.message))
    elif evento.HasField("task"):
        tarea = evento.task
        print("Respuesta tipo task:", TaskState.Name(tarea.status.state))
        textos.extend(get_artifact_text(a) for a in tarea.artifacts)

print("\nRespuesta A2A:\n", "\n".join(filter(None, textos)))

await cliente.close()
await http_a2a.aclose()


Tarjeta: agente_rodrigo -> http://127.0.0.1:2025/a2a/097095cf-e5e1-52ab-b240-14eaa9b5d64c
Respuesta tipo task: TASK_STATE_COMPLETED

Respuesta A2A:
 ¡Hola! Qué temazo me tocas. Si reviso mis apuntes sobre el **caso real del carro de compras de Falabella**, saqué aprendizajes directos de trincheras con alto tráfico multi-país y miles de usuarios concurrentes. 

Según mis apuntes, esto fue lo que aprendí:
1. **Separar bien los contratos:** Vital para que los microservicios se comuniquen sin romper la compatibilidad hacia atrás.
2. **Dimensionar el caché:** Usar Redis correctamente para aliviar la base de datos y bajar drásticamente la latencia.
3. **Diseñar para fallar de forma controlada:** En arquitecturas multi-tenant (montadas en GCP con Docker y Kubernetes), si un servicio se cae, el sistema entero no puede irse a negro.
4. **Medir todo lo que importa:** Implementar dashboards de observabilidad para tener visibilidad real en tiempo real.

*(Eso es exactamente lo que registré en mis 

## Preparar mi tarjeta para la Etapa 3

Antes de que el supervisor elija entre los especialistas, cada perfil actualiza el nombre y la
descripción de su asistente. El servidor A2A usa esos campos para publicar la Agent Card y el
supervisor los lee como **datos** para enrutar. Esta celda se ejecuta una vez por perfil; es
idempotente.

In [ ]:
# Prepara la tarjeta de Rodrigo para la Etapa 3.
# El servidor A2A toma el nombre y la descripción del asistente para armar la Agent Card.
# Esta celda es idempotente: puedes ejecutarla más de una vez.
NOMBRE_TARJETA = "Rodrigo"
DESCRIPCION_TARJETA = (
    "Especialista en arquitectura backend (Java/Spring Boot, microservicios y APIs REST: "
    "contratos, resiliencia, caché y observabilidad) y en diseño de agentes de IA "
    "(tool calling, orquestación multiagente, RAG y MCP). Explica con honestidad lo que sabe "
    "y lo que no; no inventa datos ni ejecuta acciones reales."
)

await lg.assistants.update(ASSISTANT_ID, name=NOMBRE_TARJETA, description=DESCRIPCION_TARJETA)
asistente_actualizado = await lg.assistants.get(ASSISTANT_ID)
print("Tarjeta preparada:")
print(json.dumps({
    "name": asistente_actualizado.get("name"),
    "description": asistente_actualizado.get("description"),
    "graph_id": asistente_actualizado.get("graph_id"),
    "assistant_id": asistente_actualizado.get("assistant_id"),
}, indent=2, ensure_ascii=False))


## Etapa 3 · Conversación entre especialistas

El notebook de esta etapa vive en la raíz del repositorio, porque orquesta a los dos perfiles:
[`notebooks/a2a-test.ipynb`](../../notebooks/a2a-test.ipynb).

Allí se levantan los dos servidores (Rodrigo en 2025 y Christopher en 2026), se descubren ambas
tarjetas, se prueba el router con una pregunta de cada especialidad y se ejecuta la conversación
de tres turnos.